In [ ]:
import os
import pickle
import pandas as pd
from pathlib import Path

In [ ]:
# Set this having downloaded some pickles from
# https://drive.google.com/drive/u/0/folders/18QSHI1y3KKTmZR4hT3Brn7sLyRQjwZAP
pickle_path = "~/data/diverse_tm/"
psych_path = '~/code/lsoc/data/evals/psy-100.csv'  # set this

In [ ]:
pickle_path = Path(pickle_path).expanduser()
files = list(pickle_path.glob("*.pkl"))
files = [str(s) for s in files]
tail = '_task_list_1.pkl'
models = [os.path.basename(f)[:-len(tail)].lower() for f in files]
print(f"{len(models)} models available.")

In [ ]:
targets = ['01-ai/Yi-6B',
 '01-ai/Yi-6B-Chat',
 'bigscience/T0pp',
 'BioMistral/BioMistral-7B',
 'databricks/dolly-v2-12b',
 'databricks/dolly-v2-7b',
 'databricks/dolly-v2-3b',
 'deepseek-ai/deepseek-coder-6.7b-instruct',
 'deepseek-ai/DeepSeek-R1-Distill-Llama-8B',
 'EleutherAI/gpt-j-6b',
 'EleutherAI/pythia-12b',
 'EleutherAI/pythia-14m',
 'EleutherAI/pythia-160m',
 'EleutherAI/pythia-1b',
 'EleutherAI/pythia-2.8b',
 'EleutherAI/pythia-31m',
 'EleutherAI/pythia-410m',
 'EleutherAI/pythia-6.9b',
 'EleutherAI/pythia-70m',
 'epfl-llm/meditron-7b',
 'google/gemma-2-2b',
 'google/gemma-2-9b',
 'google/gemma-2-9b-it',
 'google/gemma-2b',
 'google/gemma-2b-it',
 'google/gemma-7b',
 'google/gemma-7b-it',
 'ibm-granite/granite-3.1-2b-base',
 'ibm-granite/granite-3.1-2b-instruct',
 'ibm-granite/granite-3.1-8b-base',
 'ibm-granite/granite-3.1-8b-instruct',
 'lmsys/vicuna-13b-v1.3',
 'lmsys/vicuna-7b-v1.3',
 'maritaca-ai/sabia-7b',
 'meta-llama/Llama-2-13b-hf',
 'meta-llama/Llama-2-7b-hf',
 'meta-llama/Llama-3.2-1B',
 'meta-llama/Llama-3.2-1B-Instruct',
 'meta-llama/Llama-3.2-3B',
 'meta-llama/Llama-3.2-3B-Instruct',
 'meta-llama/Meta-Llama-3-8B',
 'microsoft/phi-2',
 'microsoft/Phi-3-medium-4k-instruct',
 'microsoft/Phi-3-small-8k-instruct',
 'mistralai/Mistral-7B-Instruct-v0.3',
 'mistralai/Mistral-7B-v0.1',
 'mistralai/Mistral-Nemo-Base-2407',
 'Qwen/Qwen1.5-14B',
 'Qwen/Qwen1.5-14B-Chat',
 'Qwen/Qwen1.5-7B',
 'Qwen/Qwen1.5-7B-Chat',
 'Qwen/Qwen2.5-7B-Instruct',
 'sail/Sailor-7B',
 'sail/Sailor-7B-Chat',
 'scb10x/llama-3-typhoon-v1.5-8b',
 'scb10x/llama-3-typhoon-v1.5-8b-instruct',
 'scb10x/typhoon-7b',
 'stabilityai/stablelm-base-alpha-3b',
 'stabilityai/stablelm-base-alpha-7b',
 'tiiuae/falcon-7b']
targets = [t.lower() for t in targets]
file_to_model = {k.replace("/", "-"): k for k in targets}

In [ ]:
# introspect a file to make a spec
filename = files[0]

with open(filename, 'rb') as f:
   data = pickle.load(f)

priority = [
    'acc,none',
    'em,none',
    'exact_match,strict-match', # or flexible-match
    'mcc,none',
    'exact,none',
    'bleu_acc,none'
]

results = data['results']
for g in data['groups']:
    if g in results:
        del results[g]

higher_is_better = data['higher_is_better']

spec = {}

for eval in results:
    for p in priority:
        if p in results[eval]:
            p_type = p.split(",")[0]
            assert higher_is_better[eval][p_type]
            spec[eval] = p
            break
    else:
        print(k, results[k].keys())
    #out = {k: results[k]['acc,none'] for k in results}

spec

In [ ]:
# # Load the evals using this spec
# rows = []
# for filename, tail in zip(files, models):
#     print(f"Loading {filename}...")
#     with open(filename, 'rb') as f:
#         data = pickle.load(f)
    
#     results = data['results']
#     row = {eval: results[eval][stat] for eval, stat in spec.items()}
#     row.update({"model": file_to_model[tail]})
#     rows.append(row)

# evals = pd.DataFrame(rows).set_index("model")
# evals

In [ ]:
# Load and adjust Project's context-aggregated pile losses to match Timaeus model naming scheme
map_df = pd.read_csv(os.path.expanduser('~/data/model_list.csv'))
mapping = map_df.set_index('name')['url'].to_dict()
mapping = {k.replace(" ", "_"):v.lower().strip() for k, v in mapping.items()}

# Now some of the new models have george's naming scheme
for t in targets:
    name = t.split("/")[1]
    mapping[name] = t
    
mapping = {v.lower():k.lower() for k, v in inv_mapping.items()}

mapping


In [ ]:
p100 = pd.read_csv(os.path.expanduser(psych_path))  # augmented
p100["Model"] = p100["Model"].map(mapping)
p100.set_index("Model", inplace=True)
# # with open('pile100.pkl', 'wb') as f:
# #    pickle.dump(pile, f)
# with open('pile100.pkl', 'rb') as f:
#    pile = pickle.load(f)
pile = p100
p100


In [ ]:
extra_url = list(set(targets) - set(pile.index))
extra_names = [f.split("/")[1] for f in extra_url]
col_names = ["name","url","num_parameters"]

extra_df = pd.DataFrame({
   'name': extra_names,
   'url': extra_url,
   'num_parameters': [1 for _ in extra_url],
})

extra_df.to_csv("extras.csv", index=False)
extra_df  # what are we missing

In [ ]:
# Load the evals
evals = pd.read_csv("~/data/lsoc1_evals_full.csv").set_index("model")
evals

In [ ]:
# Find the common ground
common_idx = pile.index.intersection(evals.index)
pile_cropped = pile.loc[common_idx]
evals_cropped = evals.loc[common_idx]
pile_cropped

In [ ]:
# Save these for further use - note, already in repo
# with open(os.path.expanduser('~/code/lsoc/data/aligned.pkl'), 'wb') as f:
#    pickle.dump({"evals": evals_cropped, "pile": pile_cropped}, f)